# Modelo:

Vamos implementar um hierarquical mixture of experts aqui

A idéia é fazer modelos para ser muito bons em esxtrair características maiores, para posteriormente extrair características menores.

Realizando a segmentação dos insetos foi observada uma pequena diminuição na qualidade do modelo. O contexto geral da imagem da indícios sobre qual inseto está sendo classificado.

In [ ]:
class Classificador_Hierarquico:
    def __init__(self):
        self.classificador_ordem = None
        self.classificador_familia = {
            'Thysanoptera': None, 'Coleoptera': None,
            'Diptera': None, 'Ixodida': None, 'Megaloptera': None, 'Phasmida': None, 'Hymenoptera': None,
            'Psocodea': None, 'Araneae': None, 'Mesostigmata': None, 'Symphyla_order_incertae_sedis': None, 
            'Stylommatophora': None, 'Julida': None, 'Sarcoptiformes': None, 'Siphonaptera': None, 'Isopoda': None,
            'Polydesmida': None, 'Trombidiformes': None, 'Orthoptera': None, 'Dermaptera': None, 'Neuroptera': None, 'Odonata': None,
            'Lepidoptera': None, 'Hemiptera': None, 'Blattodea': None, 'Embioptera': None, 'Mantodea': None, 'Symphypleona': None
        }
    
    def load(self):
        #.../modelos

        self.classificador_ordem = YOLO("/home/bradas/Documentos/github/ml-engineering/modelos/yolo-nano-ordem/train/weights/best.pt")
        
        for ordem in self.classificador_familia.keys():
            path = f"/home/bradas/Documentos/github/ml-engineering/modelos/yolo-nano-familia-{ordem}/runs/classify/train/weights/best.pt"
            modelo = YOLO(path)
            self.classificador_familia[ordem] = modelo

    def _resultado(self, classificador, path):
        results = classificador.predict(path, imgsz=224, device=0)
        ordem = results[0].names[results[0].probs.top1]
        confianca = results[0].probs.top1conf
        velocidade = results[0].speed

        confianca_top5 = results[0].probs.top5conf
        nomes = list(map(lambda x: results[0].names[x], results[0].probs.top5))
        top5 = {}
        for i in range(0,5):
            top5[nomes[i]] = confianca_top5[i]
        return ordem, confianca, velocidade, top5

    
    def predict(self, path):
        # retornar top1, top5, speed de cada um
        ordem, confianca, velocidade, top5 = self._resultado(self.classificador_ordem, path)
        print(f"Ordem inferida: {ordem}, confiança: {confianca}")
        print(velocidade)

        print("Top 5 Ordem: ")
        for i, (key, value) in enumerate(top5.items()):
            print(f" [{i+1}] | {key} | {value}")

        familia, confianca, velocidade, top5 = self._resultado(self.classificador_familia[ordem], path)
        print(f"Familia inferida: {familia}, confiança: {confianca}")
        print(velocidade)

        print("Top 5 Familia: ")
        for i, (key, value) in enumerate(top5.items()):
            print(f" [{i+1}] | {key} | {value}")

    def fit():
        pass

    def prepare_data():
        pass

    def val(self, val_dir: str, split_folder: str = 'val', zero_division=0):
        """
            Espera a estrutura de pastas:
            val_dir/
            ├── Coleoptera/
            │     └── val/
            │         ├── Carabidae/
            │         └── Scarabaeidae/
            └── Diptera/
                    └── val/
                        └── Culicidae/
        """
        val_path = Path(val_dir)

        y_true_ordem, y_pred_ordem = [], []
        y_true_familia, y_pred_familia = [], []

        # Percorre cada pasta de Ordem
        for ordem_dir in val_path.iterdir():
            if not ordem_dir.is_dir():
                continue
            ordem_real = ordem_dir.name

            # 2. Entra na subpasta 'val' (ex: val_dir/Coleoptera/val)
            split_dir = ordem_dir / split_folder
            if not split_dir.exists() or not split_dir.is_dir():
                continue

      # Percorre cada pasta de Família dentro da Ordem
            for familia_dir in split_dir.iterdir():
                if not familia_dir.is_dir():
                    continue
                familia_real = familia_dir.name

                # Aponta para a subpasta 'val' (ou 'test') dentro da família

                # Pega todas as imagens dentro de familia_dir/val/
                imagens = [p for p in familia_dir.glob('*') if p.is_file()]
                if not imagens:
                    continue

                # 1. Predição em lote no classificador de ORDEM
                res_ordem = self.classificador_ordem.predict(
                    source=imagens, verbose=False
                )

            # Agrupa imagens pela ordem predita para rodar os modelos de família em lote
                imagens_por_ordem_predita = {}
                for img_path, r_ord in zip(imagens, res_ordem):
                    ordem_pred = r_ord.names[r_ord.probs.top1]

                    y_true_ordem.append(ordem_real)
                    y_pred_ordem.append(ordem_pred)

                    imagens_por_ordem_predita.setdefault(ordem_pred, []).append(img_path)

            # 2. Predição em lote no classificador de FAMÍLIA correspondente
                for ordem_pred, imgs_grupo in imagens_por_ordem_predita.items():
                    modelo_fam = self.classificador_familia.get(ordem_pred)

                    if modelo_fam is not None:
                        res_fam = modelo_fam.predict(source=imgs_grupo, verbose=False)
                        for r_fam in res_fam:
                            familia_pred = r_fam.names[r_fam.probs.top1]
                            y_true_familia.append(familia_real)
                            y_pred_familia.append(familia_pred)
                    else:
                        # Caso a ordem predita não tenha um modelo de família associado
                        for _ in imgs_grupo:
                            y_true_familia.append(familia_real)
                            y_pred_familia.append('SEM_MODELO_FAMILIA')
                            
        print('=== RELATÓRIO: ORDEM ===')
        print(
            classification_report(
                y_true_ordem, y_pred_ordem, digits=4, zero_division=zero_division
            )
        )

        print('\n=== RELATÓRIO: FAMÍLIA (PIPELINE HIERÁRQUICO) ===')
        print(
            classification_report(
                y_true_familia,
                y_pred_familia,
                digits=4,
                zero_division=zero_division,
            )
        )

        return {
            'f1_ordem_macro': f1_score(
                y_true_ordem,
                y_pred_ordem,
                average='macro',
                zero_division=zero_division,
            ),
            'f1_ordem_weighted': f1_score(
                y_true_ordem,
                y_pred_ordem,
                average='weighted',
                zero_division=zero_division,
            ),
            'f1_familia_macro': f1_score(
                y_true_familia,
                y_pred_familia,
                average='macro',
                zero_division=zero_division,
            ),
            'f1_familia_weighted': f1_score(
                y_true_familia,
                y_pred_familia,
                average='weighted',
                zero_division=zero_division,
            ),
        }


In [8]:
ch = Classificador_Hierarquico()
ch.load()

In [ ]:
ch.predict('/home/bradas/Downloads/insetos/familia/familia_yolo/val/Noctuidae/1 (56).jpg')

In [9]:
ch.val('/home/bradas/Downloads/insetos/familia/familia_yolo_toy_hierarquico')

=== RELATÓRIO: ORDEM ===
                               precision    recall  f1-score   support

                      Araneae     0.8800    0.8148    0.8462        27
                    Blattodea     0.7619    0.5517    0.6400        29
                   Coleoptera     0.9110    0.9458    0.9281      1071
                   Dermaptera     0.8571    0.5455    0.6667        11
                      Diptera     0.7958    0.7525    0.7735       202
                   Embioptera     0.0000    0.0000    0.0000         1
                    Hemiptera     0.7069    0.7500    0.7278       328
                  Hymenoptera     0.8127    0.8282    0.8204       262
                      Isopoda     0.8571    0.6000    0.7059        10
                      Ixodida     0.5000    0.3333    0.4000        12
                       Julida     1.0000    0.6667    0.8000         3
                  Lepidoptera     0.8521    0.8748    0.8633       527
                     Mantodea     0.6667    0.6667 

{'f1_ordem_macro': 0.5329144714413354,
 'f1_ordem_weighted': 0.8366753846974131,
 'f1_familia_macro': 0.31220062124093884,
 'f1_familia_weighted': 0.6492840736985969}

In [1]:
%pip install ultralytics
%pip install -U ultralytics

Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 10.7 MB/s eta 0:00:00 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.9/102.9 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: ultralytics-platform
    Found existing installation: ultralytics-platform 0.1.94
    Uninstalling ultralytics-platform-0.1.94:
      Successfully uninstalled ultralytics-platform-0.1.94
  Attempting uninstall: ultralytics-thop
    Found existing installation: ultralytics-thop 2.2.3
    Uninstalling ultralytics-thop-2.2.3:
      Successfully uninstalled ultralytics-thop-2.2.3
  Attempting uninstall: ultralytics
    Found existing installation: ultralytics 8.4.188
    Uninstalling ultralytics-8.4.188:
      Successfully uninstalled ultralytics-8.4.188
Note: you may need to restart the kernel to use updated packages.


In [2]:
# imports
from ultralytics import YOLO
import os
import shutil
import gc
import torch
from pathlib import Path
from sklearn.metrics import classification_report, f1_score

In [ ]:
#Vamos comparar modelos, um deles segmentado e outro não para ver se a segmentação ajuda na calssificação ou piora.
model = YOLO('yolo26n-cls.pt')

# Executa o fine-tuning no dataset personalizado
results = model.train(
    data='/home/bradas/Downloads/ordem_yolo_toy',
    epochs=100,
    imgsz=224,
    batch=64,
    workers=4,
    device=0,
    amp=True,
    cache=False,
)
results

In [ ]:
# Carrega modelo salvo: teste no alvo não segmentado
model = YOLO('/home/bradas/Documentos/github/ml-engineering/runs/classify/train-3/weights/best.pt')

results = model.predict(
    source='/home/bradas/Downloads/ordem/Hymenoptera/0 (1).jpg',
    imgsz=224,
    device=0,  
    show =True
)
results

In [4]:
# esse é o modelo com segmentação
model = YOLO('yolo26n-cls.pt')

# Executa o fine-tuning no dataset personalizado
results = model.train(
    data='/home/bradas/Downloads/ordem_yolo_toy_seg',
    epochs=100,
    imgsz=224,
    batch=64,
    workers=4,
    device=0,
    amp=True,
    cache=False,
)
results

New https://pypi.org/project/ultralytics/8.4.189 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.188 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5773MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/bradas/Downloads/ordem_yolo_toy_seg, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, 

/home/bradas/Documentos/github/ml-engineering/.venv/lib/python3.12/site-packages/triton/backends/nvidia/driver.c:9:10: fatal error: Python.h: Arquivo ou diretório inexistente
    9 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.


train: /home/bradas/Downloads/ordem_yolo_toy_seg/train... found 12186 images in 28 classes ✅ 
val: /home/bradas/Downloads/ordem_yolo_toy_seg/val... found 2465 images in 28 classes ✅ 
test: None...
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 20/20 32.2it/s 0.6s0.0s
                   all      0.832      0.968
Speed: 0.1ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/bradas/Documentos/github/ml-engineering/runs/classify/train-2


ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x77bd5b230d70>
curves: []
curves_results: []
fitness: 0.8995943069458008
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8316429853439331, 'metrics/accuracy_top5': 0.9675456285476685, 'fitness': 0.8995943069458008}
save_dir: PosixPath('/home/bradas/Documentos/github/ml-engineering/runs/classify/train-2')
speed: {'preprocess': 0.10049187991857765, 'inference': 0.12736067951304414, 'loss': 3.248559814947704e-05, 'postprocess': 7.384908751734064e-05}
top1: 0.8316429853439331
top5: 0.9675456285476685

In [5]:
# Carrega modelo salvo: teste no alvo não segmentado
modelo_yolo = YOLO('/home/bradas/Documentos/github/ml-engineering/modelos/yolo-nano/train/weights/best.pt')
modelo_yolo_seg = YOLO('/home/bradas/Documentos/github/ml-engineering/modelos/yolo-nano-seg/classify/train-2/weights/best.pt')

In [6]:
print("Sem segmentar:")
print(modelo_yolo.val())
print("="*50)
print("Com segmentação:")
print(modelo_yolo_seg.val())

Sem segmentar:
Ultralytics 8.4.188 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5773MiB)
YOLO26n-cls summary (fused): 47 layers, 1,561,892 parameters, 0 gradients, 3.2 GFLOPs
train: /home/bradas/Downloads/ordem_yolo_toy/train... found 12186 images in 28 classes ✅ 
val: /home/bradas/Downloads/ordem_yolo_toy/val... found 2465 images in 28 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 83.8±21.8 MB/s, size: 12.9 KB)
val: Scanning /home/bradas/Downloads/ordem_yolo_toy/val... 2465 images, 0 corrupt: 100% ━━━━━━━━━━━━ 2465/2465 689.3Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 155/155 78.2it/s 2.0s0.1ss
                   all      0.851      0.974
Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/bradas/Documentos/github/ml-engineering/runs/classify/val
ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralyti

## Segmentação não é uma boa, vamos então criar uma comparação entre uma arquitetura hierárquica com a arquitetura normal

nesse primeiro momento vamos fazer com os toy datasets (6% do dataset real) com o menor modelo (nano), depois que a arquitetura e o código estiverem prontos vamos treinar em modelos maiores e no dataset completo

In [ ]:
# modelo de arquitetura normal para classificação de família
# basta treinar o yolo para identificar as famílias (264 classes)

# esse é o modelo com segmentação
model = YOLO('yolo26n-cls.pt')

# Executa o fine-tuning no dataset personalizado
results = model.train(
    data='/home/bradas/Downloads/insetos/familia/familia_yolo_toy',
    epochs=100,
    imgsz=224,
    batch=64,
    workers=4,
    device=0,
    amp=True,
    cache=False,
)
results

In [1]:
# treinar vários modelos com menos classes:
ordens = {'Thysanoptera': ['Aeolothripidae', 'Thripidae', 'Phlaeothripidae'],
 'Coleoptera': ['Byturidae', 'Carabidae', 'Attelabidae', 'Buprestidae', 'Phalacridae',
  'Coccinellidae', 'Mycetophagidae', 'Ptinidae', 'Bostrichidae', 'Chrysomelidae', 'Brentidae', 'Trogossitidae',
  'Dermestidae', 'Laemophloeidae', 'Elateridae', 'Staphylinidae', 'Tenebrionidae', 'Cerambycidae', 'Nitidulidae',
  'Cantharidae', 'Scarabaeidae', 'Oedemeridae', 'Cleridae', 'Silvanidae', 'Erotylidae', 'Anthicidae',
  'Meloidae', 'Cicindelidae', 'Passalidae', 'Curculionidae', 'Anthribidae', 'Cryptophagidae', 'Latridiidae'],
 'Diptera': ['Stratiomyidae',
  'Drosophilidae', 'Calliphoridae', 'Piophilidae', 'Tachinidae', 'Anthomyiidae', 'Syrphidae', 'Chloropidae',
  'Tabanidae', 'Hippoboscidae', 'Agromyzidae', 'Sciaridae', 'Fanniidae', 'Oestridae', 'Ulidiidae',
  'Asilidae', 'Culicidae', 'Ephydridae', 'Psychodidae', 'Tipulidae', 'Dolichopodidae', 'Sarcophagidae',
  'Bombyliidae', 'Tephritidae', 'Cecidomyiidae', 'Simuliidae', 'Muscidae', 'Lonchaeidae', 'Polleniidae'],
 'Ixodida': ['Ixodidae', 'Argasidae'],
 'Megaloptera': ['Corydalidae'],
 'Phasmida': ['Diapheromeridae', 'Pseudophasmatidae'],
 'Hymenoptera': ['Crabronidae', 'Mymaridae', 'Agaonidae',
  'Formicidae', 'Siricidae', 'Bembicidae', 'Dryinidae', 'Megachilidae', 'Andrenidae', 'Chrysididae',
  'Tenthredinidae', 'Cynipidae', 'Scelionidae', 'Cephidae', 'Ichneumonidae', 'Pompilidae', 'Figitidae',
  'Ampulicidae', 'Apidae', 'Eurytomidae', 'Cimbicidae', 'Pamphiliidae', 'Vespidae', 'Diplolepididae',
  'Anaxyelidae', 'Evaniidae', 'Scoliidae', 'Diprionidae', 'Sphecidae', 'Braconidae', 'Halictidae'],
 'Psocodea': ['Philopteridae', 'Menoponidae', 'Pthiridae', 'Linognathidae', 'Haematopinidae',
  'Trinotonidae', 'Hoplopleuridae', 'Lachesillidae', 'Trichodectidae', 'Polyplacidae', 'Trogiidae', 'Liposcelididae'],
 'Araneae': ['Desidae',
  'Sparassidae', 'Araneidae', 'Gnaphosidae', 'Theridiidae', 'Oxyopidae', 'Trechaleidae', 'Sicariidae'],
 'Mesostigmata': ['Dermanyssidae', 'Macronyssidae'],
 'Symphyla_order_incertae_sedis': ['Scutigerellidae'],
 'Stylommatophora': ['Milacidae', 'Spiraxidae', 'Agriolimacidae', 'Helicidae', 'Limacidae'],
 'Julida': ['Julidae'],
 'Sarcoptiformes': ['Acaridae', 'Glycyphagidae', 'Psoroptidae', 'Sarcoptidae', 'Pyroglyphidae', 'Carpoglyphidae'],
 'Siphonaptera': ['Leptopsyllidae', 'Pulicidae', 'Ceratophyllidae', 'Tungidae'],
 'Isopoda': ['Chaetiliidae', 'Asellidae'],
 'Polydesmida': ['Paradoxosomatidae'],
 'Trombidiformes': ['Tetranychidae', 'Eriophyidae', 'Tarsonemidae', 'Demodicidae', 'Tenuipalpidae', 'Penthaleidae'],
 'Orthoptera': ['Trigonidiidae', 'Gryllotalpidae',
  'Oecanthidae', 'Stenopelmatidae', 'Pyrgomorphidae', 'Acrididae', 'Romaleidae', 'Gryllidae', 'Tettigoniidae'],
 'Dermaptera': ['Chelisochidae', 'Forficulidae', 'Anisolabididae', 'Labiduridae'],
 'Neuroptera': ['Myrmeleontidae', 'Hemerobiidae', 'Chrysopidae'],
 'Odonata': ['Aeshnidae'],
 'Lepidoptera': ['Euteliidae', 'Cosmopterigidae', 'Nolidae', 'Erebidae', 'Lasiocampidae', 'Yponomeutidae', 'Hesperiidae',
  'Ypsolophidae', 'Saturniidae', 'Noctuidae', 'Geometridae', 'Pieridae', 'Nepticulidae', 'Nymphalidae',
  'Pyralidae', 'Bombycidae', 'Cossidae', 'Galacticidae', 'Heliozelidae', 'Elachistidae', 'Bucculatricidae',
  'Depressariidae', 'Lycaenidae', 'Plutellidae', 'Argyresthiidae', 'Oecophoridae', 'Bedelliidae',
  'Glyphipterigidae', 'Gracillariidae', 'Sphingidae', 'Prodoxidae', 'Riodinidae', 'Limacodidae', 'Psychidae',
  'Papilionidae', 'Choreutidae', 'Coleophoridae', 'Pterolonchidae', 'Tortricidae', 'Dudgeoneidae', 'Crambidae',
  'Tineidae', 'Sesiidae', 'Gelechiidae', 'Incurvariidae', 'Notodontidae', 'Zygaenidae', 'Schreckensteiniidae'],
 'Hemiptera': ['Tingidae',
  'Aphrophoridae', 'Ischnorhinidae', 'Clastopteridae', 'Nabidae', 'Lygaeidae', 'Membracidae', 'Geocoridae',
  'Diaspididae', 'Anthocoridae', 'Fulgoridae', 'Flatidae', 'Cicadidae', 'Gelastocoridae', 'Scutelleridae',
  'Reduviidae', 'Pentatomidae', 'Cimicidae', 'Triozidae', 'Coreidae', 'Adelgidae', 'Miridae',
  'Blissidae','Coccidae', 'Rhopalidae', 'Aphididae', 'Thyreocoridae', 'Pyrrhocoridae', 'Pseudococcidae',
  'Cydnidae', 'Cicadellidae', 'Delphacidae', 'Aleyrodidae', 'Gerridae', 'Psyllidae', 'Monophlebidae'],
 'Blattodea': ['Heterotermitidae', 'Archotermopsidae', 'Kalotermitidae', 'Pseudophyllodromiidae', 'Blattellidae',
  'Blaberidae', 'Ectobiidae', 'Corydiidae', 'Rhinotermitidae', 'Termitidae', 'Blattidae'],
 'Embioptera': ['Oligotomidae'],
 'Mantodea': ['Mantidae'],
 'Symphypleona': ['Bourletiellidae']}

def agrupa_familia_por_ordem(ordem, origem):
    destino = f"{origem}_hierarquico/{ordem}"
    os.makedirs(destino, exist_ok=True)
    
    for pasta in os.listdir(origem+"/train"):
        if pasta in ordens[ordem]:
            print(f"copiando {pasta}")
            shutil.copytree(origem+f"/train/{pasta}", destino+f"/train/{pasta}")

    for pasta in os.listdir(origem+"/val"):
        if pasta in ordens[ordem]:
            print(f"copiando {pasta}")
            shutil.copytree(origem+f"/val/{pasta}", destino+f"/val/{pasta}")

def prepara_conjuntos(origem):
    for ordem in ordens.keys():
        agrupa_familia_por_ordem(ordem, origem)

    

In [ ]:
prepara_conjuntos("/home/bradas/Downloads/insetos/familia/familia_yolo_toy")

In [ ]:
def treina_modelos(path, yolo_model, name):
    lista = os.listdir(path)
    
    for i, ordem in enumerate(lista):
        print(f"[ {i} | {len(lista)} ] - Iniciando treino para: {ordem}")
        
        model = YOLO(yolo_model)

        model.train(
            data=os.path.join(path, ordem),
            epochs=100,
            imgsz=224,
            batch=64,
            workers=10,
            device=0,
            amp=True,
            cache=True,
        )

        # ATENÇÃO: Ao executar essa função, garanta que a pasta runs não exista para mover corretamente
        
        print("Movendo resultados")
        destino = f"/home/bradas/Documentos/github/ml-engineering/modelos/modelos-hierarquico-{name}/yolo-{name}-{ordem}"
        os.makedirs(destino, exist_ok=True)
        
        origem_runs = "/home/bradas/Documentos/github/ml-engineering/runs"
        origem_weights = "/home/bradas/Documentos/github/ml-engineering/weights"
        
        if os.path.exists(origem_runs):
            shutil.move(origem_runs, destino + "/runs")
        if os.path.exists(origem_weights):
            shutil.move(origem_weights, destino + "/weights")
            
        del model
        if hasattr(torch, "cuda") and torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        print(f"Memória liberada após o treino de {ordem}\n")

In [ ]:
treina_modelos("/home/bradas/Downloads/insetos/familia/familia_yolo_toy_hierarquico", 'yolo26n-cls.pt', "nano-toy")

In [7]:
model = YOLO('/home/bradas/Documentos/github/ml-engineering/modelos/yolo-nano-familia-Araneae/runs/classify/train/weights/best.pt')

results = model.predict(
    source='/home/bradas/Downloads/insetos/ordem/ordem_yolo/train/Araneae/3 (9).jpg',
    imgsz=224,
    device=0,  
    show =True
)
results


image 1/1 /home/bradas/Downloads/insetos/ordem/ordem_yolo/train/Araneae/3 (9).jpg: 224x224 Sparassidae 0.75, Araneidae 0.22, Desidae 0.01, Gnaphosidae 0.01, Oxyopidae 0.01, 1.6ms
Speed: 3.3ms preprocess, 1.6ms inference, 0.0ms postprocess per image at shape (1, 3, 224, 224)


[ultralytics.engine.results.Results object with attributes:
 
 boxes: None
 depth: None
 keypoints: None
 masks: None
 names: {0: 'Araneidae', 1: 'Desidae', 2: 'Gnaphosidae', 3: 'Oxyopidae', 4: 'Sicariidae', 5: 'Sparassidae', 6: 'Theridiidae', 7: 'Trechaleidae'}
 obb: None
 orig_img: array([[[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        ...,
 
        [[255, 255, 255],
         [255, 255, 255],
         [255, 255, 255],
         ...,
         [255, 255, 255],
         [255, 255, 255],
         [255, 255, 255]],
 
        [

In [24]:
familia = results[0].names[results[0].probs.top1]
confianca = results[0].probs.top1conf
velocidade = results[0].speed
print(f"Familia inferida: {familia}, confiança: {confianca}")
print(velocidade)

print("Top 5 Familia: ")
confianca = results[0].probs.top5conf
nomes = list(map(lambda x: results[0].names[x], results[0].probs.top5))
for i in range(0,5):
    print(f" [{i+1}] | {nomes[i]} | {confianca[i]}")

Familia inferida: Sparassidae, confiança: 0.7506929636001587
{'preprocess': 3.2583460006208043, 'inference': 1.5954690006765304, 'postprocess': 0.02172399945266079}
Top 5 Familia: 
 [1] | Sparassidae | 0.7506929636001587
 [2] | Araneidae | 0.21713340282440186
 [3] | Desidae | 0.012965542264282703
 [4] | Gnaphosidae | 0.009084745310246944
 [5] | Oxyopidae | 0.007796425372362137


## Treinando modelos completos

Vamos treinar o dataset localmente. (GPU em núvem não funcionou)

In [ ]:
# modelo de arquitetura normal para classificação de família
# basta treinar o yolo para identificar as famílias (264 classes)

# esse é o modelo com segmentação
model = YOLO('yolo26s-cls.pt')

# Executa o fine-tuning no dataset personalizado
results = model.train(
    data='/home/bradas/Downloads/insetos/familia/familia_yolo',
    epochs=100,
    imgsz=224,
    batch=64,
    workers=10,
    device=0,
    amp=True,
    cache=True,
)
results

In [2]:
model = YOLO('/home/bradas/Documentos/github/ml-engineering/modelos/yolo-small-familia/weights/best.pt')

metrics = model.val(
    data='/home/bradas/Downloads/insetos/familia/familia_yolo',
    batch=64,
    plots=True,
    project="resultados/validacao",
    name='yolo-small-familia'
)

print(f"Acurácia Top-1: {metrics.top1:.4f}")
print(f"Acurácia Top-5: {metrics.top5:.4f}")

Ultralytics 8.4.211 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5773MiB)
YOLO26s-cls summary (fused): 47 layers, 5,772,312 parameters, 0 gradients, 12.1 GFLOPs
train: /home/bradas/Downloads/insetos/familia/familia_yolo/train... found 195120 images in 264/264 classes ✅ 
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val... found 48898 images in 264/264 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 53.8±32.4 MB/s, size: 31.9 KB)
val: Scanning /home/bradas/Downloads/insetos/familia/familia_yolo/val... 48898 images, 0 corrupt: 100% ━━━━━━━━━━━━ 48898/48898 10.3Git/s 0.0s
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val/Apidae/20260501_102814.jpg: corrupt JPEG restored and saved
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val/Apidae/20260519_160023.jpg: corrupt JPEG restored and saved
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val/Carabidae/20260519_162347(1).jpg: corrupt JPEG re

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


               classes   top1_acc   top5_acc: 76% ━━━━━━━━━─── 585/765 21.5it/s 29.6s<8.4ss

Corrupt JPEG data: 104 extraneous bytes before marker 0xd9


               classes   top1_acc   top5_acc: 90% ━━━━━━━━━━╸─ 692/765 24.0it/s 34.3s<3.0s

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 765/765 19.8it/s 38.6s0.1s
                   all       0.91       0.97
Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/bradas/Documentos/github/ml-engineering/runs/classify/resultados/validacao/yolo-small-familia
Acurácia Top-1: 0.9097
Acurácia Top-5: 0.9699


In [3]:
model = YOLO('/home/bradas/Documentos/github/ml-engineering/modelos/yolo-nano-familia/weights/best.pt')

metrics = model.val(
    data='/home/bradas/Downloads/insetos/familia/familia_yolo',
    batch=64,
    plots=True,
    project="resultados/validacao",
    name='yolo-small-familia'
)

print(f"Acurácia Top-1: {metrics.top1:.4f}")
print(f"Acurácia Top-5: {metrics.top5:.4f}")

Ultralytics 8.4.211 🚀 Python-3.12.3 torch-2.14.1+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 5773MiB)
YOLO26n-cls summary (fused): 47 layers, 1,864,208 parameters, 0 gradients, 3.2 GFLOPs
train: /home/bradas/Downloads/insetos/familia/familia_yolo/train... found 195120 images in 264/264 classes ✅ 
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val... found 48898 images in 264/264 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 927.8±806.2 MB/s, size: 31.9 KB)
val: Scanning /home/bradas/Downloads/insetos/familia/familia_yolo/val... 48898 images, 0 corrupt: 100% ━━━━━━━━━━━━ 48898/48898 22.8Git/s 0.0s
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val/Apidae/20260501_102814.jpg: corrupt JPEG restored and saved
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val/Apidae/20260519_160023.jpg: corrupt JPEG restored and saved
val: /home/bradas/Downloads/insetos/familia/familia_yolo/val/Carabidae/20260519_162347(1).jpg: corrupt JPEG r

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


               classes   top1_acc   top5_acc: 75% ━━━━━━━━━─── 580/765 34.5it/s 30.0s<5.4ss

Corrupt JPEG data: 104 extraneous bytes before marker 0xd9


               classes   top1_acc   top5_acc: 90% ━━━━━━━━━━╸─ 691/765 33.9it/s 35.1s<2.2s

libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50


               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 765/765 19.6it/s 39.0s0.1s
                   all      0.781      0.888
Speed: 0.1ms preprocess, 0.2ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/bradas/Documentos/github/ml-engineering/runs/classify/resultados/validacao/yolo-small-familia-2
Acurácia Top-1: 0.7807
Acurácia Top-5: 0.8884


In [3]:
# separando por hierarquia
prepara_conjuntos("/home/bradas/Downloads/insetos/familia/familia_yolo")

copiando Aeolothripidae
copiando Thripidae
copiando Phlaeothripidae
copiando Aeolothripidae
copiando Thripidae
copiando Phlaeothripidae
copiando Byturidae
copiando Carabidae
copiando Attelabidae
copiando Buprestidae
copiando Phalacridae
copiando Coccinellidae
copiando Mycetophagidae
copiando Ptinidae
copiando Bostrichidae
copiando Chrysomelidae
copiando Brentidae
copiando Trogossitidae
copiando Dermestidae
copiando Laemophloeidae
copiando Elateridae
copiando Staphylinidae
copiando Tenebrionidae
copiando Cerambycidae
copiando Nitidulidae
copiando Cantharidae
copiando Scarabaeidae
copiando Oedemeridae
copiando Cleridae
copiando Silvanidae
copiando Erotylidae
copiando Anthicidae
copiando Meloidae
copiando Cicindelidae
copiando Passalidae
copiando Curculionidae
copiando Anthribidae
copiando Cryptophagidae
copiando Latridiidae
copiando Byturidae
copiando Carabidae
copiando Attelabidae
copiando Buprestidae
copiando Phalacridae
copiando Coccinellidae
copiando Mycetophagidae
copiando Ptinidae


In [ ]:
#TODO: Retreinar o yolo NANO com todos os dados             | FEITO |
#TODO: treinar o yolo MEDIUM com todos os dados             | FEITO |
#TODO: treinar o yolo SMALL com todos os dados              | FEITO |
#TODO: Retreinar o yolo NANO hierárquico todos os dados     | FEITO |
#TODO: treinar o yolo SMALL hierárquico todos os dados
#TODO: treinar o yolo MEDIUM hierárquico com todos os dados | Falta ordem |